# 01 Preprocess: smoothing, glitches, direction check
Runs on the clean tables from `00_preprocess` (`data/processed/clean/`) and writes them back with new columns. Run 00 first.

What this adds:
- **Combine fix:** 551 clips have ball rows under the same `event_id`. 00 grouped by `event_id` only, so `step_dist` and `a_signed` were stitched across ball and player in those clips. Recomputed here grouped by `event_id` and `entity_type`.
- **Smoothing:** Savitzky-Golay speed and acceleration (`s_sg`, `a_sg`; window 7, order 2). Use `a_sg` for features.
- **Glitch flags:** `glitch_frame`, `glitch_track`. Drop glitch tracks before features.
- **Direction check** for the flip done in 00.

In [1]:
from shared import preprocess as pp
from shared.config import PROCESSED
import pandas as pd
import sys
sys.path.insert(0, '.')

CLEAN = PROCESSED / 'clean'
CT_KEYS = ['event_id', 'entity_type']
GAME_KEYS = ['game_id', 'play_id', 'nfl_id']
Q = [0, .001, .01, .5, .99, .999, 1]

## Combine tracking

In [2]:
ct = pd.read_parquet(CLEAN / 'combine_tracking.parquet')
ct = pp.add_kinematics(ct, CT_KEYS)      # redo with the fixed grouping
ct = pp.smooth_kinematics(ct, CT_KEYS)
ct = pp.flag_glitches(ct, CT_KEYS)
ct.shape

(463189, 20)

In [3]:
ct.head()

,draft_year,event_id,nfl_id,entity_type,time,drill_type,drill_name,attempt,x,y,s,a,dis,dir,step_dist,a_signed,s_sg,a_sg,glitch_frame,glitch_track
0,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.500,FORTY_YARD_DASH,FORTY_YARD_DASH,1,10.900000,-1.433333,0.586667,3.335603,0.001111,63.725788,NaN,NaN,0.573016,3.616667,False,False
1,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.600,FORTY_YARD_DASH,FORTY_YARD_DASH,1,11.000000,-1.433333,0.943333,3.692252,0.010000,70.346176,0.100000,3.566667,0.947619,3.875397,False,False
2,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.700,FORTY_YARD_DASH,FORTY_YARD_DASH,1,11.100000,-1.433333,1.310000,4.038059,0.010000,73.594005,0.100000,3.666667,1.348095,4.134127,False,False
3,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.800,FORTY_YARD_DASH,FORTY_YARD_DASH,1,11.266667,-1.333333,1.793333,4.309581,0.029523,71.800835,0.194365,4.833333,1.774444,4.392857,False,False
4,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.900,FORTY_YARD_DASH,FORTY_YARD_DASH,1,11.466667,-1.266667,2.236667,4.540093,0.040246,73.995960,0.210819,4.433333,2.235873,4.633333,False,False


Check 1 again with the fixed grouping: step distance / (s x 0.1) near 1.

In [4]:
p = ct[ct['entity_type'].eq('PLAYER')]
chk = p.dropna(subset=['step_dist'])
chk = chk[chk['s'] > 1]
(chk['step_dist'] / (chk['s'] * 0.1)).median()

np.float64(0.9963574995608789)

Check 7: acceleration for player rows. Expect extremes around -10 to +10 yd/s^2.

In [5]:
pd.DataFrame({'a_signed': p['a_signed'].quantile(
    Q), 'a_sg': p['a_sg'].quantile(Q)}).round(2)

,a_signed,a_sg
0.000,-10.63,-9.82
0.001,-8.12,-6.94
0.010,-5.83,-4.83
0.500,0.93,0.91
0.990,6.13,5.56
0.999,7.89,6.88
1.000,11.87,10.25


Check 8: smoothing window on the 40. Peaks should barely move between 5, 7 and 9 frames.

In [6]:
forty = p[p['drill_name'].eq('FORTY_YARD_DASH')]
sens = {}
for w in (5, 7, 9):
    k = pp.smooth_kinematics(forty, CT_KEYS, window=w)
    sens[w] = k.groupby('event_id').agg(peak_accel=(
        'a_sg', 'max'), peak_speed=('s_sg', 'max')).median()
pd.DataFrame(sens).round(2)

,5,7,9
peak_accel,5.54,5.43,5.33
peak_speed,10.59,10.59,10.59


Check 9: glitch clips and clips too short to smooth.

In [7]:
pd.Series({'player clips': p['event_id'].nunique(),
           'glitch clips': p.loc[p['glitch_track'], 'event_id'].nunique(),
           'too short to smooth': p.groupby('event_id')['a_sg'].apply(lambda a: a.isna().all()).sum()})

player clips           6310
glitch clips              0
too short to smooth       0
dtype: int64

In [8]:
ct.to_parquet(CLEAN / 'combine_tracking.parquet', index=False)

## Game tracking, one season at a time

In [9]:
plays = pd.read_parquet(CLEAN / 'player_play.parquet')
summary = {}
for yr in (2023, 2024, 2025):
    g = pd.read_parquet(CLEAN / f'game_tracking_{yr}.parquet')
    g = pp.smooth_kinematics(g, GAME_KEYS)
    g = pp.flag_glitches(g, GAME_KEYS)
    good = g[~g['glitch_track']]
    direction = pp.check_direction(g, plays)
    summary[yr] = {
        'tracks': g.groupby(GAME_KEYS).ngroups,
        'glitch tracks': g[g['glitch_track']].groupby(GAME_KEYS).ngroups,
        'a_signed min (all)': g['a_signed'].min(),
        'a_sg min (no glitch)': good['a_sg'].min(),
        'a_sg p0.1 (no glitch)': good['a_sg'].quantile(.001),
        'a_sg p99.9 (no glitch)': good['a_sg'].quantile(.999),
        'a_sg max (no glitch)': good['a_sg'].max(),
        'route runners gaining x': direction['route runners gaining x'],
        'pass blockers losing x': direction['pass blockers losing x'],
    }
    g.to_parquet(CLEAN / f'game_tracking_{yr}.parquet', index=False)
    print(yr, 'done', len(g))
    del g, good

2023 done 3665443
2024 done 7827944
2025 done 11160903


Checks 6, 8, 9. Expect:
- glitch tracks about 0.2% of tracks
- `a_sg` without glitches within about -15 to +15, 0.1% tails near +/-7
- route runners gaining x and pass blockers losing x both above 0.9

In [10]:
pd.DataFrame(summary).round(3)

,2023,2024,2025
tracks,50911.000,108720.000,155313.000
glitch tracks,112.000,212.000,239.000
a_signed min (all),-77.400,-83.200,-81.600
a_sg min (no glitch),-13.854,-14.636,-16.693
a_sg p0.1 (no glitch),-6.639,-6.504,-6.407
a_sg p99.9 (no glitch),6.675,5.821,5.821
a_sg max (no glitch),13.818,10.918,11.525
route runners gaining x,0.953,0.940,0.943
pass blockers losing x,0.920,0.922,0.911
